# Agri-Smart: Padi Viability Predictor
## Notebook 02 — Model Training & Evaluation
**Course:** COMP6577001 — Machine Learning | BINUS University
**Dataset:** java_padi_dataset_v3.csv (896 rows, 14 climate features, 112 kabupaten, 2018–2025)
**Approach:** Binary Classification — Viable / Not Viable
**Models compared:** Logistic Regression, Random Forest, SVM, Decision Tree, KNN,
Gradient Boosting, XGBoost, Extra Trees, Soft Voting Ensemble

---

## 1. Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
import warnings
warnings.filterwarnings('ignore')

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    ExtraTreesClassifier, VotingClassifier,
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, f1_score, accuracy_score, roc_curve,
)
from sklearn.ensemble import HistGradientBoostingClassifier

# XGBoost is also excellent but requires `brew install libomp` on Mac.
# HistGradientBoostingClassifier is sklearn's equivalent with no native deps.
try:
    from xgboost import XGBClassifier
    _XGBOOST = True
    print("XGBoost available")
except Exception:
    _XGBOOST = False
    print("XGBoost not available (install libomp via brew). Using HistGradientBoosting instead.")

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
print("Libraries loaded successfully")

## 2. Load Dataset (v3 — 896 rows, 2018–2025)

In [ ]:
df = pd.read_csv('../data/processed/java_padi_dataset_v3.csv')

print(f"Shape         : {df.shape}")
print(f"Kabupaten     : {df['kabupaten'].nunique()}")
print(f"Provinces     : {sorted(df['province'].unique())}")
print(f"Years         : {sorted(df['year'].unique())}")
print(f"Missing values: {df.isnull().sum().sum()}")
df.head()

## 3. Feature Engineering

### Overview of features built
| Group | Features | Count |
|---|---|---|
| Climate (raw) | 14 ERA5-Land variables | 14 |
| Kabupaten encoding | kab_mean, kab_std, kab_median, kab_min, kab_max | 5 |
| Lag features | yield_lag1, yield_lag2 | 2 |
| Climate anomaly z-scores | (value − kab_avg) / kab_std per climate feature | 14 |
| **Total (extended)** | | **35** |

All kab stats and anomaly stats are computed from **training data only** to prevent leakage.

In [ ]:
CLIMATE_14 = [
    'temperature_mean_c', 'temperature_max_c', 'temperature_min_c',
    'rainfall_mm_year', 'precip_hours_day', 'humidity_pct',
    'sunshine_hrs_day', 'shortwave_radiation', 'et0_mm_day',
    'vapour_pressure_def', 'wind_speed',
    'soil_moisture_0_7cm', 'soil_moisture_7_28cm', 'soil_temperature',
]
KAB_STAT_COLS = ['kab_mean', 'kab_std', 'kab_median', 'kab_min', 'kab_max']
ANOMALY_COLS  = [f'{c}_anomaly' for c in CLIMATE_14]

# ── 3.1  Sort & compute lag features on full df (shift is within-kabupaten) ──
df = df.sort_values(['kabupaten', 'year']).reset_index(drop=True)
df['yield_lag1'] = df.groupby('kabupaten')['yield_ton_ha'].shift(1)
df['yield_lag2'] = df.groupby('kabupaten')['yield_ton_ha'].shift(2)

# ── 3.2  Time-based split: train 2018–2024, test 2025 ────────────────────────
train = df[df['year'] <= 2024].copy()
test  = df[df['year'] == 2025].copy()
print(f"Train: {len(train)} rows | Test: {len(test)} rows")

# ── 3.3  Kabupaten stats — from TRAIN only ────────────────────────────────────
kab_stats = (
    train.groupby('kabupaten')['yield_ton_ha']
         .agg(['mean','std','median','min','max'])
         .rename(columns={'mean':'kab_mean','std':'kab_std',
                          'median':'kab_median','min':'kab_min','max':'kab_max'})
)
for col in KAB_STAT_COLS:
    train[col] = train['kabupaten'].map(kab_stats[col])
    test[col]  = test['kabupaten'].map(kab_stats[col])

# Fill lag NaNs with kab_mean (first-year rows have no lag)
for split in [train, test]:
    m1 = split['yield_lag1'].isna()
    m2 = split['yield_lag2'].isna()
    split.loc[m1, 'yield_lag1'] = split.loc[m1, 'kab_mean']
    split.loc[m2, 'yield_lag2'] = split.loc[m2, 'kab_mean']

# ── 3.4  Climate anomaly z-scores — from TRAIN only ──────────────────────────
kab_clim_mean = train.groupby('kabupaten')[CLIMATE_14].mean()
kab_clim_std  = train.groupby('kabupaten')[CLIMATE_14].std().fillna(1.0)

for split in [train, test]:
    for col in CLIMATE_14:
        means = split['kabupaten'].map(kab_clim_mean[col])
        stds  = split['kabupaten'].map(kab_clim_std[col]).fillna(1.0)
        split[f'{col}_anomaly'] = (split[col] - means) / stds

# ── 3.5  Feature sets ─────────────────────────────────────────────────────────
BASE_FEATURES     = CLIMATE_14 + KAB_STAT_COLS + ['yield_lag1', 'yield_lag2']  # 21
EXTENDED_FEATURES = BASE_FEATURES + ANOMALY_COLS                                # 35

print(f"Base features    : {len(BASE_FEATURES)}")
print(f"Extended features: {len(EXTENDED_FEATURES)}")
print(f"\nFirst 5 anomaly cols: {ANOMALY_COLS[:5]}")

## 4. Binary Label Creation

**Threshold** = median of TRAINING yields (2018–2024) — no leakage into 2025.
- **Not Viable (0):** yield < threshold → bottom half of Java producers
- **Viable (1):** yield ≥ threshold → top half of Java producers

Display tiers (33rd/67th percentile of full dataset):
- 🔴 Bad: yield < low\_threshold
- 🟡 Moderate: low\_threshold ≤ yield < high\_threshold
- 🟢 Great: yield ≥ high\_threshold

In [ ]:
threshold      = train['yield_ton_ha'].median()
low_threshold  = df['yield_ton_ha'].quantile(0.33)
high_threshold = df['yield_ton_ha'].quantile(0.67)

train['viable'] = (train['yield_ton_ha'] >= threshold).astype(int)
test['viable']  = (test['yield_ton_ha']  >= threshold).astype(int)

print("=== Label Distribution ===")
print(f"Threshold (train median) : {threshold:.4f} ton/ha")
print(f"Display — Bad      : yield < {low_threshold:.4f}")
print(f"Display — Moderate : {low_threshold:.4f} ≤ yield < {high_threshold:.4f}")
print(f"Display — Great    : yield ≥ {high_threshold:.4f}")
print()
print(f"Train — Not Viable: {(train['viable']==0).sum()}  Viable: {(train['viable']==1).sum()}")
print(f"Test  — Not Viable: {(test['viable']==0).sum()}   Viable: {(test['viable']==1).sum()}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(train[train['viable']==0]['yield_ton_ha'], bins=20,
             alpha=0.7, color='#ef5350', label='Not Viable', edgecolor='white')
axes[0].hist(train[train['viable']==1]['yield_ton_ha'], bins=20,
             alpha=0.7, color='#4caf50', label='Viable', edgecolor='white')
axes[0].axvline(threshold, color='navy', linestyle='--', linewidth=2,
                label=f'Median: {threshold:.3f}')
axes[0].set_title('Yield Distribution by Class (Training Data)')
axes[0].set_xlabel('Yield (ton/ha)')
axes[0].legend()

counts = train['viable'].value_counts().sort_index()
axes[1].bar(['Not Viable (0)', 'Viable (1)'], counts.values,
            color=['#ef5350', '#4caf50'], edgecolor='white', width=0.5)
for i, v in enumerate(counts.values):
    axes[1].text(i, v + 3, f'{v} ({v/len(train)*100:.1f}%)', ha='center', fontsize=11)
axes[1].set_title('Class Distribution (Training Set)')
axes[1].set_ylabel('Count')
plt.tight_layout()
plt.savefig('../data/processed/label_distribution.png', dpi=130, bbox_inches='tight')
plt.show()

## 5. Model Comparison — 5-Fold Stratified CV (Training Set, Base 21 Features)

Cross-validation on training data only (2018–2024). Final test on 2025 is in Section 8.

In [ ]:
X_train_base = train[BASE_FEATURES]
y_train      = train['viable']

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def make_pipe(clf):
    return Pipeline([('sc', StandardScaler()), ('clf', clf)])

models = {
    'Logistic Regression':  make_pipe(LogisticRegression(C=1, max_iter=1000, random_state=42)),
    'Random Forest':        make_pipe(RandomForestClassifier(n_estimators=300, min_samples_leaf=2, random_state=42)),
    'SVM (RBF)':            make_pipe(SVC(C=10, kernel='rbf', gamma='scale', probability=True, random_state=42)),
    'Decision Tree':        make_pipe(DecisionTreeClassifier(max_depth=6, min_samples_leaf=5, random_state=42)),
    'KNN':                  make_pipe(KNeighborsClassifier(n_neighbors=5, weights='distance')),
    'Gradient Boosting':    make_pipe(GradientBoostingClassifier(n_estimators=300, learning_rate=0.05, max_depth=3, random_state=42)),
    'Hist Gradient Boost':  make_pipe(HistGradientBoostingClassifier(
                                          max_iter=300, learning_rate=0.05,
                                          max_depth=4, random_state=42)),
    'Extra Trees':          make_pipe(ExtraTreesClassifier(n_estimators=300, min_samples_leaf=2, random_state=42)),
}

# Optionally add XGBoost if available
if _XGBOOST:
    models['XGBoost'] = make_pipe(XGBClassifier(
        n_estimators=300, learning_rate=0.05, max_depth=4,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric='logloss', random_state=42, verbosity=0))

cv_results = {}
print(f"{'Model':<22} {'Accuracy':>10} {'F1':>10} {'AUC-ROC':>10} {'Precision':>10} {'Recall':>10}")
print('-' * 78)

for name, pipe in models.items():
    cv = cross_validate(pipe, X_train_base, y_train, cv=skf,
                        scoring=['accuracy', 'f1', 'roc_auc', 'precision', 'recall'])
    cv_results[name] = {k: cv[f'test_{k}'].mean() for k in ['accuracy','f1','roc_auc','precision','recall']}
    r = cv_results[name]
    print(f"{name:<22} {r['accuracy']:>10.4f} {r['f1']:>10.4f} {r['roc_auc']:>10.4f} "
          f"{r['precision']:>10.4f} {r['recall']:>10.4f}")

best_cv_name = max(cv_results, key=lambda x: cv_results[x]['f1'])
print(f"\nRandom baseline : 0.5000")
print(f"Best (CV F1)    : {best_cv_name} — F1={cv_results[best_cv_name]['f1']:.4f}")

## 6. Soft Voting Ensemble (LR + RF + GB + XGBoost)

Soft voting averages predicted probabilities. Best when member models make *different* errors.

In [ ]:
_ensemble_members = [
    ('lr',  LogisticRegression(C=1, max_iter=1000, random_state=42)),
    ('rf',  RandomForestClassifier(n_estimators=300, min_samples_leaf=2, random_state=42)),
    ('gb',  GradientBoostingClassifier(n_estimators=300, learning_rate=0.05, max_depth=3, random_state=42)),
    ('hgb', HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, max_depth=4, random_state=42)),
]
if _XGBOOST:
    _ensemble_members.append(('xgb', XGBClassifier(
        n_estimators=300, learning_rate=0.05, max_depth=4,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric='logloss', random_state=42, verbosity=0)))

voting_pipe = Pipeline([
    ('sc', StandardScaler()),
    ('clf', VotingClassifier(estimators=_ensemble_members, voting='soft')),
])

cv_v = cross_validate(voting_pipe, X_train_base, y_train, cv=skf,
                      scoring=['accuracy', 'f1', 'roc_auc', 'precision', 'recall'])
cv_results['Voting Ensemble'] = {k: cv_v[f'test_{k}'].mean() for k in ['accuracy','f1','roc_auc','precision','recall']}
r = cv_results['Voting Ensemble']
print("=== Soft Voting Ensemble (LR + RF + GB + XGBoost) ===")
print(f"Accuracy : {r['accuracy']:.4f}")
print(f"F1-Score : {r['f1']:.4f}")
print(f"AUC-ROC  : {r['roc_auc']:.4f}")
print(f"Precision: {r['precision']:.4f}")
print(f"Recall   : {r['recall']:.4f}")

## 7. Extended Features Experiment — Climate Anomaly Z-Scores

Adding 14 climate anomaly features (deviation from kabupaten's historical average)
to the top 3 CV models. Anomalies capture "unusually hot/wet year" signals.

In [ ]:
X_train_ext = train[EXTENDED_FEATURES]

top3 = sorted(cv_results, key=lambda x: cv_results[x]['f1'], reverse=True)[:3]
print("Testing extended features on top-3 CV models:")
print(f"  {top3}")
print()

ext_results = {}
print(f"{'Model (extended)':<28} {'F1 (base)':>12} {'F1 (ext)':>12} {'ΔΔΔF1':>10} {'AUC (ext)':>12}")
print('-' * 78)

for name in top3:
    if name == 'Voting Ensemble':
        continue
    pipe_ext = make_pipe(models[name].named_steps['clf'].__class__(
        **models[name].named_steps['clf'].get_params()))
    cv_e = cross_validate(pipe_ext, X_train_ext, y_train, cv=skf,
                          scoring=['f1', 'roc_auc'])
    f1_base = cv_results[name]['f1']
    f1_ext  = cv_e['test_f1'].mean()
    auc_ext = cv_e['test_roc_auc'].mean()
    ext_results[name] = {'f1': f1_ext, 'auc': auc_ext}
    print(f"{name:<28} {f1_base:>12.4f} {f1_ext:>12.4f} {f1_ext-f1_base:>+10.4f} {auc_ext:>12.4f}")

print("\nConclusion: if extended F1 > base F1, add anomaly features to final model.")

## 8. Final Evaluation — Time-Based Test Set (2025)

Train on all 2018–2024 data, evaluate on held-out 2025.
This is the authoritative metric reported in the app and PPT.

In [ ]:
X_test_base = test[BASE_FEATURES]
X_test_ext  = test[EXTENDED_FEATURES]
y_test      = test['viable']

# Determine best feature set from experiment
# Use extended if any top-3 model improved; otherwise use base
use_ext = any(
    ext_results.get(n, {}).get('f1', 0) > cv_results[n]['f1']
    for n in ext_results
)
print(f"Use extended features: {use_ext}")

# Candidates to evaluate on test set
candidates = {
    'Logistic Regression': Pipeline([
        ('sc', StandardScaler()),
        ('clf', LogisticRegression(C=1, max_iter=1000, random_state=42)),
    ]),
    'Random Forest': Pipeline([
        ('sc', StandardScaler()),
        ('clf', RandomForestClassifier(n_estimators=300, min_samples_leaf=2, random_state=42)),
    ]),
    'Gradient Boosting': Pipeline([
        ('sc', StandardScaler()),
        ('clf', GradientBoostingClassifier(n_estimators=300, learning_rate=0.05, max_depth=3, random_state=42)),
    ]),
    'Hist Gradient Boost': Pipeline([
        ('sc', StandardScaler()),
        ('clf', HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, max_depth=4, random_state=42)),
    ]),
    'Extra Trees': Pipeline([
        ('sc', StandardScaler()),
        ('clf', ExtraTreesClassifier(n_estimators=300, min_samples_leaf=2, random_state=42)),
    ]),
    'Voting Ensemble': voting_pipe,
}
# XGBoost only if available (requires `brew install libomp` on Mac)
if _XGBOOST:
    candidates['XGBoost'] = Pipeline([
        ('sc', StandardScaler()),
        ('clf', XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=4,
                               subsample=0.8, colsample_bytree=0.8,
                               eval_metric='logloss', random_state=42, verbosity=0)),
    ])

X_tr = train[EXTENDED_FEATURES] if use_ext else train[BASE_FEATURES]
X_te = X_test_ext if use_ext else X_test_base
feat_label = 'Extended (35)' if use_ext else 'Base (21)'

test_results = {}
print(f"\nFeature set: {feat_label}")
print(f"{'Model':<22} {'Accuracy':>10} {'F1':>10} {'AUC-ROC':>10}")
print('-' * 55)

for name, pipe in candidates.items():
    pipe.fit(X_tr, y_train)
    y_pred = pipe.predict(X_te)
    y_prob = pipe.predict_proba(X_te)[:, 1]
    test_results[name] = {
        'accuracy': accuracy_score(y_test, y_pred),
        'f1':       f1_score(y_test, y_pred),
        'auc':      roc_auc_score(y_test, y_prob),
        'pipe':     pipe,
    }
    r = test_results[name]
    print(f"{name:<22} {r['accuracy']:>10.4f} {r['f1']:>10.4f} {r['auc']:>10.4f}")

best_test_name = max(test_results, key=lambda x: test_results[x]['f1'])
print(f"\n★ Best on Test 2025: {best_test_name}")
print(f"  Accuracy : {test_results[best_test_name]['accuracy']*100:.2f}%")
print(f"  F1-Score : {test_results[best_test_name]['f1']:.4f}")
print(f"  AUC-ROC  : {test_results[best_test_name]['auc']:.4f}")

FINAL_MODEL = test_results[best_test_name]['pipe']
FINAL_NAME  = best_test_name
FINAL_FEATS = EXTENDED_FEATURES if use_ext else BASE_FEATURES

In [ ]:
# Full classification report for best model
y_pred_best = FINAL_MODEL.predict(X_te)
print(f"=== Classification Report — {FINAL_NAME} ===")
print(classification_report(y_test, y_pred_best, target_names=['Not Viable', 'Viable']))

## 9. Evaluation Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ── Confusion matrix ──────────────────────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred_best)
ConfusionMatrixDisplay(cm, display_labels=['Not Viable', 'Viable']).plot(
    ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title(f'Confusion Matrix\n{FINAL_NAME} (Test 2025)', fontsize=12)

# ── Model comparison bar chart ────────────────────────────────────────────────
# Include all cv_results models for comparison
all_names = [n for n in cv_results if n != 'Voting Ensemble'] + ['Voting Ensemble']
f1_cv  = [cv_results[n]['f1']  for n in all_names]
auc_cv = [cv_results[n]['roc_auc'] for n in all_names]
x = np.arange(len(all_names))
w = 0.35
axes[1].bar(x - w/2, f1_cv,  w, label='F1-Score (CV)', color='#42a5f5', alpha=0.85, edgecolor='white')
axes[1].bar(x + w/2, auc_cv, w, label='AUC-ROC (CV)', color='#66bb6a', alpha=0.85, edgecolor='white')
axes[1].axhline(0.5, color='red', linestyle='--', linewidth=1.5, label='Random baseline')
axes[1].set_xticks(x)
axes[1].set_xticklabels([n.replace(' ', '\n') for n in all_names], fontsize=7)
axes[1].set_ylim(0.4, 1.0)
axes[1].set_title('Model Comparison (5-Fold CV, Training Set)', fontsize=11)
axes[1].set_ylabel('Score')
axes[1].legend(fontsize=9)

# ── ROC Curve ─────────────────────────────────────────────────────────────────
y_prob_best = FINAL_MODEL.predict_proba(X_te)[:, 1]
fpr, tpr, _ = roc_curve(y_test, y_prob_best)
auc_val = roc_auc_score(y_test, y_prob_best)
axes[2].plot(fpr, tpr, color='#1565C0', linewidth=2.5,
             label=f'ROC Curve (AUC = {auc_val:.4f})')
axes[2].plot([0, 1], [0, 1], '--', color='gray', linewidth=1.5, label='Random Baseline')
axes[2].fill_between(fpr, tpr, alpha=0.1, color='#1565C0')
axes[2].set_xlabel('False Positive Rate')
axes[2].set_ylabel('True Positive Rate')
axes[2].set_title(f'ROC Curve — {FINAL_NAME} (Test 2025)', fontsize=12)
axes[2].legend()

plt.tight_layout()
plt.savefig('../data/processed/model_evaluation_plots.png', dpi=130, bbox_inches='tight')
plt.show()
print("Saved: ../data/processed/model_evaluation_plots.png")

## 10. Feature Importance

In [ ]:
# Use Random Forest for model-agnostic importance insight
rf_imp = RandomForestClassifier(n_estimators=300, min_samples_leaf=2, random_state=42)
rf_imp.fit(X_tr, y_train)

feat_imp = pd.Series(rf_imp.feature_importances_, index=FINAL_FEATS).sort_values(ascending=True)
top15 = feat_imp.tail(15)

colors = ['#4caf50' if v >= feat_imp.median() else '#90CAF9' for v in top15.values]
plt.figure(figsize=(10, 7))
top15.plot(kind='barh', color=colors, edgecolor='white')
plt.title('Top 15 Feature Importances (Random Forest, Training Set)', fontsize=13)
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('../data/processed/feature_importance_final.png', dpi=130, bbox_inches='tight')
plt.show()

print("Top 10 most important features:")
print(feat_imp.sort_values(ascending=False).head(10).round(4).to_string())

## 11. Save Best Model

> **Note:** If the best model here differs from the one in `app/model.pkl`
> (produced by `train_model.py`), run `python train_model.py` from the project root
> to regenerate with the full pipeline including province stats maps.
> The notebook saves a compatible model and metadata for reference.

In [ ]:
# Refit best model on full training set
FINAL_MODEL.fit(X_tr, y_train)
joblib.dump(FINAL_MODEL, '../app/model.pkl')
print(f"Model saved: ../app/model.pkl  ({FINAL_NAME})")

# Rebuild kab_yield_map and kab_stats_map
kab_yield_map  = kab_stats['kab_mean'].to_dict()
kab_stats_map  = kab_stats.to_dict(orient='index')

# Province proxy kab_stats (mean of kabupaten-level kab_stats per province)
kab_prov   = train[['kabupaten','province']].drop_duplicates().set_index('kabupaten')
kab_stats_df = kab_stats.copy()
kab_stats_df['province'] = kab_stats_df.index.map(kab_prov['province'])
prov_agg   = kab_stats_df.groupby('province')[KAB_STAT_COLS].mean()
prov_stats_map = {
    prov: {col: round(row[col], 4) for col in KAB_STAT_COLS}
    for prov, row in prov_agg.iterrows()
}

# Kabupaten climate means for anomaly inference (if using extended features)
kab_climate_stats = {}
for kab in kab_clim_mean.index:
    kab_climate_stats[kab] = {
        col: {
            'mean': round(float(kab_clim_mean.loc[kab, col]), 6),
            'std':  round(float(kab_clim_std.loc[kab, col]), 6),
        }
        for col in CLIMATE_14
    }

y_pred_final = FINAL_MODEL.predict(X_te)
y_prob_final = FINAL_MODEL.predict_proba(X_te)[:, 1]

# Province climate stats for Option 2 anomaly computation
prov_clim_mean = train.groupby('province')[CLIMATE_14].mean()
prov_clim_std  = train.groupby('province')[CLIMATE_14].std().fillna(1.0)
prov_climate_stats = {
    prov: {
        col: {'mean': round(float(prov_clim_mean.loc[prov, col]), 6),
              'std':  round(float(prov_clim_std.loc[prov, col]), 6)}
        for col in CLIMATE_14
    }
    for prov in prov_clim_mean.index
}

meta = {
    'model_type':          'binary_classification',
    'model_name':          FINAL_NAME,
    'features':            list(FINAL_FEATS),
    'threshold':           round(float(threshold), 4),
    'low_threshold':       round(float(low_threshold), 4),
    'high_threshold':      round(float(high_threshold), 4),
    'accuracy':            round(accuracy_score(y_test, y_pred_final), 4),
    'f1_score':            round(f1_score(y_test, y_pred_final), 4),
    'auc_roc':             round(roc_auc_score(y_test, y_prob_final), 4),
    'kab_yield_map':       {k: round(v, 4) for k, v in kab_yield_map.items()},
    'kab_stats_map':       {k: {sk: round(sv, 4) for sk, sv in v.items()} for k, v in kab_stats_map.items()},
    'prov_stats_map':      prov_stats_map,
    'prov_climate_stats':  prov_climate_stats,
    'kab_climate_stats':   kab_climate_stats,
}

with open('../app/model_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)
print(f"Metadata saved: ../app/model_meta.json")

print()
print("=== FINAL SUMMARY ===")
print(f"Model    : {FINAL_NAME}")
print(f"Features : {len(FINAL_FEATS)}")
print(f"Accuracy : {meta['accuracy']*100:.2f}%")
print(f"F1-Score : {meta['f1_score']:.4f}")
print(f"AUC-ROC  : {meta['auc_roc']:.4f}")